In [2]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver

from selenium.webdriver import ActionChains

from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.common.action_chains import ActionChains

from selenium.webdriver.common.keys import Keys

from bs4 import BeautifulSoup

import pandas as pd

from time import sleep

import datetime

from pandas import ExcelWriter

import os

# %%



# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'NO FNET' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.13.0")

now=datetime.datetime.now()

filename= '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])


scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)




#------------------------------------------------ Begin_chromedriver ----------------------------------------



#Starting Chrome driver, set to download files in tempfolder



chromeOptions = webdriver.ChromeOptions()



prefs = {"plugins.always_open_pdf_externally": True,



		 "download.prompt_for_download": False,



		 "download.default_directory" : tempfolder}



chromeOptions.add_experimental_option("prefs",prefs)



driver = webdriver.Chrome(options=chromeOptions)



driver.maximize_window()



#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty

    return sqldict



def scrollinAndClick(xpath,key_press=False):

    if len(xpath) != 0 :
        for times in range(60):
            try:
                driver.find_element(By.XPATH, xpath).click()
                sleep(5)
                break

            except:

                # print(f"[ERROR] : trying {times+1}/10 to key press 'DOWN' (scrolling)")
                sleep(5)

                if key_press:                    

                    driver.find_element(By.TAG_NAME, 'body').send_keys(key_press)

        else:   
            raise Exception(f'[ERROR] : Failed scrollin Or Click on xpath element : {xpath}')


# Define a function to scroll to the bottom of the page

def scroll_to_bottom(driver):

    # Get scroll height

    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:

        # Scroll down to the bottom

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # Wait to load the page
        sleep(1)

        # Calculate new scroll height and compare with last scroll height

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:

            break

        last_height = new_height



def wait_for_pagination_update(driver, old_page_num):

    while True:

        # Wait for a short period to allow the page to update

        sleep(5)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        pagination = soup.find('div', {'aria-label': 'Load more'})

        if pagination:

            new_page_num = pagination.find('div').text.split('of')[0]

            if new_page_num != old_page_num:

                break

# Define a function to click an element by xpath

def click_element_by_xpath(driver, xpath):

    # Find the element and click

    element = driver.find_element(By.XPATH, xpath)

    element.click()



# Define a function to scroll the page until the specified element is visible

def scroll_until_element_visible(driver, element_xpath):

    # Wait for the specified element to be present in the DOM

    element = WebDriverWait(driver, 15).until(

        EC.presence_of_element_located((By.XPATH, element_xpath))

    )
    # Scroll the page until the specified element is in view

    driver.execute_script("arguments[0].scrollIntoView();", element)



def check_dowload_files(tempfolder, fileType ):

    for time in range(10):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : Failed to Download {fileType} file. Run Script again' )

# %%



#------------------------------------------------ Begin_Variable ----------------------------------------

Reports = {
    "Banking and finance":"https://www.finanstilsynet.no/en/finanstilsynets-registry/user-defined-report/?group=11",

            "Debt collection":"https://www.finanstilsynet.no/en/finanstilsynets-registry/user-defined-report/?group=5",

           "Estate agency": "https://www.finanstilsynet.no/en/finanstilsynets-registry/user-defined-report/?group=9",

           "Securities":"https://www.finanstilsynet.no/en/finanstilsynets-registry/user-defined-report/?group=4",

           "Insurance and pensions":"https://www.finanstilsynet.no/en/finanstilsynets-registry/user-defined-report/?group=2",

           "Insurance mediation":"https://www.finanstilsynet.no/en/finanstilsynets-registry/user-defined-report/?group=1",
           

           }

regdict = {"Banking and finance"       : {'Bank'                   : ['NO FNET 8', '//*[@id="BANK_chkbx"]'], 

                                           'Payment institution'     : ['NO FNET 7', '//*[@id="BETF_chkbx"]'], 

                                           'E-money institution'     : ['NO FNET 4', '//*[@id="EPENGEF_chkbx"]'], 

                                           'Holding company'         : ['NO FNET 6', '//*[@id="FINANSKONS_chkbx"]'], 

                                           'Finance company'         : ['NO FNET 5', '//*[@id="FINF_chkbx"]'], 
                                           #'Trust'                   : ['NO FNET 10', '//*[@id="FINSTIFT_chkbx"]'],  #AactulLY clicked financial foundation
                                            'Financial foundation' : ['NO FNET 10', '//*[@id="FINSTIFT_chkbx"]'], #AactulLY clicked financial foundation
                                           'Savings bank foundation' : ['NO FNET 9', '//*[@id="SPBAST_chkbx"]'],

                                           },


            "Debt collection"           : {'Agency debt collection on behalf of others'        : ['NO FNET 11', '//*[@id="FREMINKASO_chkbx"]'],

                                            'Debt collection agency - purchase and collection' : ['NO FNET 13', '//*[@id="OPPEGENINF_chkbx"]'],
                                          },

            "Insurance and pensions"    : {'Municipal pension fund'       : ['NO FNET 19', '//*[@id="KOMMPENKAS_chkbx"]'], 
                                           'Life insurance company'       : ['NO FNET 17', '//*[@id="LIVSFORSIK_chkbx"]'], 
                                           'Pension foundation'           : ['NO FNET 21', '//*[@id="PENFOND_chkbx"]'], 
                                           'Marine insurance association' : ['NO FNET 18', '//*[@id="SJØTRYGDEL_chkbx"]'], 
                                           'Non-life insurance company'   : ['NO FNET 20', '//*[@id="SKADEFORSI_chkbx"]'], 
                                           'Private pension fund'         : ['NO FNET 22', '//*[@id="PRIVPENKAS_chkbx"]'],

                                           },

            "Insurance mediation"       :{'Ancillary agent activities' : ['NO FNET 23', '//*[@id="AKSFORAGV_chkbx"]'], 

                                          'Insurance agency'           : ['NO FNET 24', '//*[@id="FORAGNTFTK_chkbx"]'], 

                                          'Insurance brokerage firm'   : ['NO FNET 26', '//*[@id="FORMGLFTK_chkbx"]'], 

                                          'Reinsurance brokerage firm' : ['NO FNET 27', '//*[@id="GFORMGLFTK_chkbx"]'],
                                         },
            
            "Securities"                :{'Investment firm'            : ['NO FNET 31', '//*[@id="FFOR_chkbx"]']},
            
            "Estate agency"             :{'Estate agency, Act adopted on 29 June 2007':['NO FNET 14','//*[@id="EIEMGLFTK_chkbx"]']}

            }


operation_type = {'Finanstilsynet':'//*[@id="virksomhet-type-norway"]',

                  'Cross-border':'//*[@id="virksomhet-type-gov"]'}       


sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],

          'Phone - Mother company': [], 'Check': []}


processdate = now.strftime('%Y-%m-%d')




#------------------------------------------------ Begin_Main ----------------------------------------

df = pd.DataFrame(sqldict)
for i, category in enumerate(Reports):
    
    links = []
    typos = []
    print(f'[INFO] : Working with category {i+1}/{len(Reports)} : {category} ')
    driver.get(Reports[category])

    try:
        # Wait until the button is visible and clickable
        accept_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "coi-banner__accept"))
        )
        
        # Click the button
        accept_button.click()
        print("[INFO:] Clicked the 'Accept all' button.")
        
    except Exception as e:
        print("Error:", e)


    driver.refresh()
    sleep(3)
    for k, typology in enumerate(regdict[category]):
        sleep(3)
        
        driver.get(Reports[category])

        reg = regdict[category][typology][0]

        xpath = regdict[category][typology][1]

        try:
            sleep(3)
            scrollinAndClick(xpath,key_press=Keys.DOWN)
        except:
            sleep(3)
            webpage_link = driver.current_url
            driver.get(webpage_link)
            scrollinAndClick(xpath,key_press=Keys.DOWN)
            

        #print(f'[INFO] : Select {len(regdict[category])} checkbox [x] ')
        print(f'[INFO] : Select {k+1} checkbox [x] {typology}')

        sleep(3)
        scrollinAndClick('//button[@aria-label="Include license provider types"]',key_press=Keys.DOWN)
        scrollinAndClick('//*[@id="Licensed"]',key_press=Keys.DOWN)
        sleep(3)
        scrollinAndClick("//a[contains(text(),'Download report as Excel')]",key_press=Keys.DOWN)
        sleep(2)
        print(f'[INFO] : Download excel file {typology}')
        sleep(3)
        check_dowload_files(tempfolder, 'xlsx')

        concat_df = pd.DataFrame(sqldict)
        for i in range(len(os.listdir(tempfolder))):
            filePath = os.path.join(tempfolder, os.listdir(tempfolder)[i])
            print(filePath)
            tempdf = pd.read_excel(filePath)
            print(f'[INFO] -- Excel file {tempdf.shape}')
            valid_rows = tempdf.iloc[:, :len(tempdf.columns)].notna().all(axis=1)
            if valid_rows.any():
                header_row_index = valid_rows.idxmax()
                print(f"First valid header row: {header_row_index}")
                tempdf.columns = tempdf.iloc[header_row_index].str.replace('\n', ' ')
                # Remove rows up to the header row
                tempdf = tempdf.iloc[header_row_index+1:].reset_index(drop=True)
            else:
                print(f"No valid header row found (first {len(tempdf.columns)} columns contain NaN in every row)")
            concat_df = pd.DataFrame(sqldict)
            tempdf = tempdf.fillna('')
            concat_df['Name'] = tempdf[tempdf.columns[0]].apply(lambda x: x.replace('\n',' ').strip())
            concat_df = concat_df.fillna('')
            concat_df['ListProcessDate'] = processdate
            concat_df['ListName'] = typology
            concat_df['RegCtry'] = reg.split(' ')[0]
            concat_df['RegCode'] = reg.split(' ')[1]
            concat_df['ListCode'] = reg.split(' ')[-1]
            concat_df['RegulationType'] = 'Regulated'
            concat_df['RegulationType'] = 'Regulated'
            concat_df['License_Type'] = tempdf[tempdf.columns[3]].apply(lambda x: x.replace('\n',' ').strip())

            concat_df['InternalID_1']=tempdf[tempdf.columns[10]].iloc[:,0]
            concat_df['InternalID_1_type'] = 'Organisation no.'
            concat_df['LEI Code']=tempdf[tempdf.columns[12]].iloc[:,0]
            concat_df['InternalID_2_type'] = 'FT number'
            concat_df['InternalID_2']=tempdf[tempdf.columns[11]].iloc[:,0]
            concat_df['RegulationDate'] = tempdf[tempdf.columns[9]]
            concat_df['City'] = tempdf[tempdf.columns[16]].iloc[:,-1]
            
            tempdf = pd.DataFrame()

            print('[INFO] -- Current File Shape: ',concat_df.shape)
            df = pd.concat([df, concat_df], axis=0)
            print('[INFO] -- Total File Shape: ',df.shape)   
            
            sleep(3)
            for rem in os.listdir(tempfolder):
                os.remove(os.path.join(tempfolder, rem))

# %%



    

Running NO FNET Web Scraping Tool v.1.13.0
[INFO] : Working with category 1/6 : Banking and finance 
Clicked the 'Accept all' button.
[INFO] : Select 1 checkbox [x] Bank
[INFO] : Download excel file Bank
[INFO] : xlsx file = ['CustomReport-2025-08-29-1734.xlsx'])
C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\NO FNET\tempfolder\CustomReport-2025-08-29-1734.xlsx
[INFO] -- Excel file (1640, 27)
First valid header row: 12
[INFO] -- Current File Shape:  (1627, 44)
[INFO] -- Total File Shape:  (1627, 44)
[INFO] : Select 2 checkbox [x] Payment institution
[INFO] : Download excel file Payment institution
[INFO] : xlsx file = ['CustomReport-2025-08-29-1735.xlsx'])
C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\NO FNET\tempfolder\CustomReport-2025-08-29-1735.xlsx
[INFO] -- Excel file (65, 27)
First valid header row: 12
[INFO] -- Current File Shape:  (52, 44)
[INFO] -- Total File Shape:  (1679, 44)
[INFO] : Select 3 checkbox [x] E-money institution
[INFO] : Download excel file E-money in

In [ ]:

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


os.chdir(scriptfolder)

# df=pd.DataFrame(sqldict)

#df = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    
    

C:\Users\wuj1\AppData\Local\Temp\10\ipykernel_16460\2553391181.py:10: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('bank_list3andend.csv')